# MCA Timestamp Example

Continuous event logging with microsecond timestamps, plus live histogram.

Uses the `mca_timestamp_1ch` bitfile: IIR highpass -> FIR -> peak detector -> histogram + event logger.

In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt
from redpitaya_control.redpitaya_dev import redpitaya_dev
from redpitaya_control.event_logger import EventLogger, unpack, load_run, fit_clock, counter_to_unix
from redpitaya_control import compute_coeff

## 1. Connect and load bitfile

In [ ]:
RP_HOST = os.environ.get("RP_HOST", "171.64.56.120")
dev = redpitaya_dev(RP_HOST, "config/mca_timestamp_1ch.json")
dev.base.load_bitfile()
rp = dev.base

## 2. Configure the MCA signal chain

In [ ]:
# IIR highpass at 10 kHz
dev.set_all_registers('iir1', compute_coeff.highpass_1st(1e4, Ts=16e-9), reset=True)

# FIR filter
dev.set_register('fir9', 'h0', 0.99)

# Peak detector
dev.set_register("peak_detector", "invert_input", 0)
dev.set_register("peak_detector", "trig_level", 0.01)
dev.set_register("peak_detector", "integration_mode", 0)
dev.set_register("peak_detector", "n_integration", 1000)
dev.set_register("peak_detector", "log_attenuation", 0)

# Histogram (register-based, 1024 bins)
dev.set_register("histogram", "offset", 0)
dev.set_register("histogram", "gain", 0.1)
dev.set_register("histogram", "band_low", 0.03)
dev.set_register("histogram", "band_high", 0.35)
dev.set_register("histogram", "pulse_width", 1024)
dev.set_register("histogram", "clear_bins", 1)
time.sleep(0.01)
dev.set_register("histogram", "clear_bins", 0)
dev.set_register("histogram", "counting_enable", 1)

## 3. Run the event logger

Records every in-band peak with a microsecond timestamp and channel-B tag.
Output: hourly `events_YYYYMMDD_HH.bin` files + `tiepoints.csv`.

In [ ]:
log = EventLogger(
    rp,
    base=0x40004000,
    bram_addr=0x41000000,
    cdma_addr=0x7E200000,
    ddr_addr=0x10000000,
    frame_len=4096,
)

log.configure(
    presc=125,            # 1 us tick @ 125 MHz
    band_low=-32768,      # full range (log everything)
    band_high=32767,
    chb_thr=0,            # channel-B threshold
    flush_ms=100,         # flush partial buffer after 100 ms
)

In [ ]:
OUTPUT_DIR = "run1"
DURATION_S = 60  # short test run; set to 3600+ for real acquisition

n = log.run(DURATION_S, output_dir=OUTPUT_DIR, tie_period_s=1.0)
print(f"Done: {n} events logged to {OUTPUT_DIR}/")

## 4. Load and unpack the event data

In [ ]:
raw = load_run(os.path.join(OUTPUT_DIR, "events_*.bin"))
ts_us, energy, chb = unpack(raw)

print(f"Events: {len(ts_us)}")
print(f"Time span: {(ts_us[-1] - ts_us[0]) / 1e6:.1f} s" if len(ts_us) > 1 else "")
print(f"ChB=1 fraction: {chb.mean():.3f}" if len(chb) else "")

## 5. Clock calibration (NTP tie-points)

In [ ]:
tp_path = os.path.join(OUTPUT_DIR, "tiepoints.csv")
a, b, ppm = fit_clock(tp_path)
print(f"Crystal drift: {ppm:+.1f} ppm  (nominal 1000 ns/tick, measured {b:.3f} ns/tick)")

t_unix = counter_to_unix(ts_us, a, b)  # absolute UNIX time in seconds

## 6. Energy spectrum (from event log)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(energy, bins=256, range=(0, 2**15), log=True, color='steelblue', edgecolor='none')
ax.set_xlabel("Energy (ADC units)")
ax.set_ylabel("Counts")
ax.set_title("Energy spectrum (from event log)")
plt.tight_layout()
plt.show()

## 7. Event rate over time

In [ ]:
if len(t_unix) > 10:
    t_rel = t_unix - t_unix[0]  # seconds from start
    bin_width = 1.0  # 1-second bins
    bins = np.arange(0, t_rel[-1] + bin_width, bin_width)
    rate, edges = np.histogram(t_rel, bins=bins)

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.step(edges[:-1], rate, where='post', color='darkorange')
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Events / s")
    ax.set_title("Event rate")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough events for a rate plot.")

## 8. Channel-B coincidence

Compare energy spectra for events with and without the channel-B tag.

In [ ]:
if len(energy) > 0 and chb.any():
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(energy[chb == 0], bins=256, range=(0, 2**15), log=True,
            alpha=0.7, label="ChB = 0", color='steelblue', edgecolor='none')
    ax.hist(energy[chb == 1], bins=256, range=(0, 2**15), log=True,
            alpha=0.7, label="ChB = 1", color='crimson', edgecolor='none')
    ax.set_xlabel("Energy (ADC units)")
    ax.set_ylabel("Counts")
    ax.set_title("Energy spectrum by channel-B tag")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No ChB=1 events (threshold may need tuning).")